# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Nov 3, 2026 - Day 20: Inference without a likelihood - ABC

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullFormatter
from matplotlib.lines import Line2D
from scipy import stats
from ipywidgets import interactive_output, IntSlider, Checkbox, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in fits 1280 px; 7.4 in beside widget controls

import threading                                    # lock matplotlib's math-text parser (slider redraws run on another thread)
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock(); _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True; _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    for w in widgets.values():
        w.layout = Layout(width='230px'); w.style = {'description_width': '95px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# ---- the toy kilonova survey (numbers from Perkins et al. 2026, arXiv:2602.13471)
F_OBS = 0.5        # fraction of the sky Rubin covers (their choice)
M_LIM = 24.0       # g-band depth 25.0, and they assume a kilonova is first seen ~1 mag past peak: 24.0 at peak
SIG_M = 0.5        # spread of kilonova peak absolute magnitudes (our choice)
R_TRUE = 56.0      # neutron-star merger rate, per Gpc^3 per year (the rate they adopt)
MU_TRUE = -14.816  # mean peak absolute g magnitude that gives their 29 kilonovae per year

def expected_count(R, mu, years=1.0):
    """Expected number of kilonovae found: the rate times the volume where they are bright enough (see the EXTRA slide)."""
    mean_rmax_cubed = 10**(3 * (M_LIM - mu) / 5 + 3) * np.exp(0.5 * (0.6 * np.log(10) * SIG_M)**2)   # pc^3
    return R * 1e-27 * F_OBS * 4 / 3 * np.pi * mean_rmax_cubed * years

def simulate(R, mu, rng, years=1.0):
    """One survey year: distances (Mpc) and apparent peak magnitudes of the kilonovae found.
    It draws only the ones bright enough to find - about 14x fewer random numbers than simulate_everything below,
    and the same answer whenever r_max stays inside 1.5 Gpc (mu fainter than about -16.5)."""
    n = rng.poisson(expected_count(R, mu, years))
    M = rng.normal(mu - 0.6 * np.log(10) * SIG_M**2, SIG_M, n)      # the found ones are brighter (last Thursday's Malmquist bias)
    r_max = 10**((M_LIM - M) / 5 + 1)                                 # pc: the farthest each one could be and still be found
    r = r_max * rng.uniform(size=n)**(1 / 3)                          # evenly spread in volume inside r_max
    return r / 1e6, M + 5 * np.log10(r) - 5

def simulate_everything(R, mu, rng, r_out=1500.0):
    """The slow way: every merger out to r_out Mpc in one year, then the magnitude cut."""
    n = rng.poisson(R * 1e-9 * F_OBS * 4 / 3 * np.pi * r_out**3)
    r = r_out * rng.uniform(size=n)**(1 / 3)                          # Mpc
    M = rng.normal(mu, SIG_M, n)
    m = M + 5 * np.log10(r * 1e6) - 5                                 # m = M + 5 log10(d / 10 pc)
    return r, M, m <= M_LIM

# ---- "the data": one simulated year at the true values (seed 0). These kilonovae are FAKE: the real sample is one.
r_all, M_all, found = simulate_everything(R_TRUE, MU_TRUE, np.random.default_rng(0))
d_obs = r_all[found]
m_obs = M_all[found] + 5 * np.log10(d_obs * 1e6) - 5
N_obs = len(d_obs)
S_obs = np.array([N_obs, np.median(m_obs), np.median(d_obs)])     # three summaries: count, median magnitude, median distance

# ---- 200,000 simulations from the prior (about 6 s; cached in data/day20_abc.npz - delete it to re-run)
rng = np.random.default_rng(20)
if os.path.exists('data/day20_abc.npz'):
    _c = np.load('data/day20_abc.npz'); log_R, mu_draw, S_sim = _c['log_R'], _c['mu'], _c['S']
else:
    n_sim = 200_000
    log_R = rng.uniform(1, 3, n_sim)            # prior: R from 10 to 1000 per Gpc^3 per year, flat in log R
    mu_draw = rng.uniform(-17, -13, n_sim)      # prior: mean peak magnitude from -17 to -13
    S_sim = np.full((n_sim, 3), np.nan)
    for i in range(n_sim):
        d, m = simulate(10**log_R[i], mu_draw[i], rng)
        S_sim[i, 0] = len(d)
        if len(d):
            S_sim[i, 1:] = np.median(m), np.median(d)
    np.savez_compressed('data/day20_abc.npz', log_R=log_R.astype(np.float32), mu=mu_draw.astype(np.float32), S=S_sim.astype(np.float32))
    _c = np.load('data/day20_abc.npz'); log_R, mu_draw, S_sim = _c['log_R'], _c['mu'], _c['S']   # use the saved copy, so a fresh run matches the cached one

TOL_MAG, TOL_DIST = 0.15, 0.05                                        # "close enough": 0.15 mag, and 5% in distance
def keep(use_mag=False, use_dist=False, tol_d=TOL_DIST):
    k = S_sim[:, 0] == N_obs                                          # the same count, exactly
    if use_mag:
        k &= np.abs(S_sim[:, 1] - S_obs[1]) <= TOL_MAG
    if use_dist:
        k &= np.abs(np.log(S_sim[:, 2] / S_obs[2])) <= tol_d
    return k
keep_N, keep_Nm, keep_Nd = keep(), keep(use_mag=True), keep(use_dist=True)


In [ ]:
# Plotting functions for the slides (a "skip" cell)
def plot_one_year():
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.6))
    ax[0].scatter(r_all[~found], M_all[~found], s=6, c='0.75', label=f'missed ({(~found).sum()})')
    ax[0].scatter(r_all[found], M_all[found], s=14, c='C0', label=f'found ({found.sum()})')
    rr = np.linspace(50, 1500, 200)
    ax[0].plot(rr, M_LIM - 5 * np.log10(rr * 1e6) + 5, 'k--', lw=1, label='the cut, $m = 24.0$')
    ax[0].set(xlabel='distance (Mpc)', ylabel='peak absolute mag $M$ (g)', ylim=(-13, -16.8), title='every merger out to 1.5 Gpc, one year')
    ax[0].legend(fontsize=10, loc='lower left')
    ax[1].hist(m_obs, bins=np.arange(21, 24.25, 0.25), color='C0', alpha=0.8)
    ax[1].set(xlabel='apparent peak mag $m$ (g)', ylabel='kilonovae', title=f'the {N_obs} found: our "data"', xlim=(21, 24.2))
    plt.tight_layout(); plt.show()

def plot_1d(eps=0, ax=None):
    rng1 = np.random.default_rng(7)
    R1 = 10**rng1.uniform(1, 3, 200_000)
    N1 = rng1.poisson(expected_count(R1, MU_TRUE))
    acc = np.abs(N1 - N_obs) <= eps
    c = expected_count(1.0, MU_TRUE)                     # kilonovae found per unit R
    grid = np.linspace(1, 130, 400)
    exact = stats.gamma(a=N_obs, scale=1 / c).pdf(grid)  # Poisson likelihood x prior flat in log R
    show = ax is None
    if show:
        fig, ax = plt.subplots(figsize=(7.4, 3.6))
    ax.hist(R1[acc], bins=np.linspace(0, 130, 53), density=True, color='C0', alpha=0.6, label=f'ABC: kept {acc.mean():.1%}')
    ax.plot(grid, exact, 'k', lw=2, label='exact posterior')
    ax.axvline(R_TRUE, color='C3', ls=':', label='true R')
    ax.set(xlabel='merger rate R (Gpc$^{-3}$ yr$^{-1}$)', ylabel='density', xlim=(0, 130), ylim=(0, 0.05),
           title=f'keep R if $|N_{{\\rm sim}} - {N_obs}| \\leq {eps}$')
    ax.legend(fontsize=10)
    if show:
        plt.tight_layout(); plt.show()

def eps_controls():
    controls_right(plot_1d, eps=IntSlider(value=0, min=0, max=20, step=1, description='ε (tolerance)'))

def _panel(ax, k, title, legend=False):
    ax.scatter(log_R[::40], mu_draw[::40], s=1, c='0.85', rasterized=True, label='drawn from the prior')
    ax.scatter(log_R[k], mu_draw[k], s=4, c='C0', alpha=0.6, label='kept')
    ax.plot(np.log10(R_TRUE), MU_TRUE, 'r*', ms=14, mec='k', label='truth')
    ax.set(xlabel='log$_{10}$ R', title=title, xlim=(1, 3), ylim=(-17, -13), xticks=[1, 1.5, 2, 2.5])
    if legend:
        handles = [Line2D([], [], ls='', marker='o', ms=5, c='0.75'), Line2D([], [], ls='', marker='o', ms=5, c='C0'),
                   Line2D([], [], ls='', marker='*', ms=10, mfc='r', mec='k')]
        ax.legend(handles, ['drawn from the prior', 'kept', 'truth'], fontsize=9, loc='lower right')

def plot_ridge():
    fig, ax = plt.subplots(figsize=(6.0, 3.9))
    _panel(ax, keep_N, f'same count ({N_obs}): {keep_N.sum()} kept', legend=True)
    ax.set_ylabel('mean peak mag $\\mu$')
    plt.tight_layout(); plt.show()

def plot_summaries():
    ok = np.isfinite(S_sim[:, 1]) & (S_sim[:, 0] >= 5)
    sel = np.where(ok)[0][::20]
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.6))
    ax[0].scatter(mu_draw[sel], S_sim[sel, 1], s=2, c='C1', alpha=0.4, rasterized=True)
    ax[0].axhline(S_obs[1], color='k', ls='--', lw=1, label='our data')
    ax[0].set(xlabel='mean peak mag $\\mu$  (brighter to the right)', ylabel='median apparent mag', title="doesn't change with $\\mu$")
    ax[1].scatter(mu_draw[sel], S_sim[sel, 2], s=2, c='C2', alpha=0.4, rasterized=True)
    ax[1].axhline(S_obs[2], color='k', ls='--', lw=1, label='our data')
    ax[1].set(xlabel='mean peak mag $\\mu$  (brighter to the right)', ylabel='median distance (Mpc)', yscale='log', title='changes with $\\mu$')
    ax[1].yaxis.set_major_formatter(ScalarFormatter()); ax[1].yaxis.set_minor_formatter(NullFormatter()); ax[1].set_yticks([200, 500, 1000])
    for a in ax:
        a.invert_xaxis(); a.legend(fontsize=10, loc='upper left')
    plt.tight_layout(); plt.show()

def plot_choose(use_mag=False, use_dist=False, tol_d_percent=5):
    k = keep(use_mag, use_dist, tol_d_percent / 100)
    fig, ax = plt.subplots(figsize=(6.4, 3.9))
    used = 'count' + (' + median mag' if use_mag else '') + (' + median distance' if use_dist else '')
    _panel(ax, k, f'{used}: {k.sum()} kept', legend=True)
    ax.set_ylabel('mean peak mag $\\mu$')
    plt.tight_layout(); plt.show()

def choose_controls():
    controls_right(plot_choose, use_mag=Checkbox(value=False, description='+ median mag'),
                   use_dist=Checkbox(value=False, description='+ median dist.'),
                   tol_d_percent=IntSlider(value=5, min=1, max=30, step=1, description='dist. tol. (%)'))

def plot_cost():
    tols = np.arange(1, 31)
    f_d = [keep(use_dist=True, tol_d=t / 100).mean() for t in tols]
    f_md = [keep(use_mag=True, use_dist=True, tol_d=t / 100).mean() for t in tols]
    fig, ax = plt.subplots(figsize=(7.4, 3.3))
    ax.axhline(keep_N.mean(), color='C0', label='count only')
    ax.plot(tols, f_d, 'C2', label='count + median distance')
    ax.plot(tols, f_md, 'C1', label='count + median mag + median distance')
    ax.set(xlabel='distance tolerance (%)', ylabel='fraction of simulations kept', yscale='log', xlim=(1, 30), ylim=(2e-5, 2e-2))
    ax.legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    return fig

def plot_three():
    fig, ax = plt.subplots(1, 3, figsize=(9.4, 3.4), sharey=True)
    _panel(ax[0], keep_N, f'count: {keep_N.sum()} kept')
    _panel(ax[1], keep_Nm, f'+ median mag: {keep_Nm.sum()}')
    _panel(ax[2], keep_Nd, f'+ median distance: {keep_Nd.sum()}', legend=True)
    ax[0].set_ylabel('mean peak mag $\\mu$')
    plt.tight_layout()
    return fig


In [ ]:
# More slide functions (a "skip" cell): match the data by hand, and ABC-SMC
from ipywidgets import FloatSlider

def plot_by_hand(log10_R=2.5, mu=-14.0, year=1):
    d, m = simulate(10**log10_R, mu, np.random.default_rng(year))
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    bins_d = np.linspace(0, 1500, 16)
    ax[0].hist(d_obs, bins=bins_d, histtype='step', color='k', lw=2, label=f'our data ({N_obs})')
    ax[0].hist(d, bins=bins_d, color='C1', alpha=0.6, label=f'this simulation ({len(d)})')
    ax[0].set(xlabel='distance (Mpc)', ylabel='kilonovae found')
    ax[0].legend(fontsize=9, loc='upper right')
    bins_m = np.arange(20, 24.25, 0.5)
    ax[1].hist(m_obs, bins=bins_m, histtype='step', color='k', lw=2)
    ax[1].hist(m, bins=bins_m, color='C1', alpha=0.6)
    ax[1].set(xlabel='apparent peak mag $m$', xlim=(20, 24.2))
    fig.suptitle(f'R = {10**log10_R:.0f},  $\\mu$ = {mu:.1f}:  {len(d)} found (we found {N_obs})', fontsize=13)
    plt.tight_layout(); plt.show()

def by_hand_controls():
    controls_right(plot_by_hand,
                   log10_R=FloatSlider(value=2.5, min=1, max=3, step=0.05, description='log10 R', readout_format='.2f'),
                   mu=FloatSlider(value=-14.0, min=-17, max=-13, step=0.1, description='μ', readout_format='.1f'),
                   year=IntSlider(value=1, min=1, max=20, step=1, description='which year'))

# ---- ABC-SMC: shrink the tolerance in steps, drawing each new theta near the ones kept last time
# (population Monte Carlo, Beaumont et al. 2009). Same match rule as before; the last step is the rejection-ABC rule.
EPS_N = [8, 4, 2, 1, 0, 0]                    # how far the count may be from ours (we picked these steps by hand)
EPS_D = [0.40, 0.25, 0.15, 0.10, 0.07, 0.05]  # how far the median distance may be (as a fraction)
N_PART = 1000                                 # kept draws per step

def run_smc(seed=5):
    rng = np.random.default_rng(seed)
    lo, hi = np.array([1.0, -17.0]), np.array([3.0, -13.0])
    gens, n_sims = [], []
    total = 0
    for t, (eN, eD) in enumerate(zip(EPS_N, EPS_D)):
        if t:
            prev, w = gens[-1], weights
            cov = 2 * np.cov(prev.T, aweights=w)
            L = np.linalg.cholesky(cov)
        theta = np.empty((N_PART, 2)); n = 0
        while n < N_PART:
            if t == 0:
                th = lo + (hi - lo) * rng.uniform(size=2)
            else:
                th = prev[rng.choice(N_PART, p=w)] + L @ rng.normal(size=2)
                if np.any(th < lo) or np.any(th > hi):
                    continue                      # outside the prior: no simulation needed
            total += 1
            d, m = simulate(10**th[0], th[1], rng)
            if abs(len(d) - N_obs) <= eN and len(d) and abs(np.log(np.median(d) / S_obs[2])) <= eD:
                theta[n] = th; n += 1
        if t == 0:
            weights = np.full(N_PART, 1 / N_PART)
        else:
            Linv = np.linalg.inv(L)
            z = (theta[:, None, :] - prev[None, :, :]) @ Linv.T          # (new, old, 2)
            kern = np.exp(-0.5 * (z**2).sum(-1)) @ w
            weights = 1 / kern; weights /= weights.sum()
        gens.append(theta); n_sims.append(total)
        if t == len(EPS_N) - 1:
            final_w = weights
    return np.array(gens), np.array(n_sims), final_w

if os.path.exists('data/day20_smc.npz'):
    _s = np.load('data/day20_smc.npz'); smc_gens, smc_nsims, smc_w = _s['gens'], _s['nsims'], _s['w']
else:
    smc_gens, smc_nsims, smc_w = run_smc()        # about 10 s
    np.savez_compressed('data/day20_smc.npz', gens=smc_gens.astype(np.float32), nsims=smc_nsims, w=smc_w)
    _s = np.load('data/day20_smc.npz'); smc_gens, smc_nsims, smc_w = _s['gens'], _s['nsims'], _s['w']

def plot_smc(step=1, ax=None):
    show = ax is None
    if show:
        fig, ax = plt.subplots(figsize=(6.4, 3.9))
    t = step - 1
    ax.scatter(log_R[::40], mu_draw[::40], s=1, c='0.85', rasterized=True)
    ax.scatter(smc_gens[t, :, 0], smc_gens[t, :, 1], s=4, c='C0', alpha=0.5)
    ax.plot(np.log10(R_TRUE), MU_TRUE, 'r*', ms=14, mec='k')
    ax.set(xlabel='log$_{10}$ R', ylabel='mean peak mag $\\mu$', xlim=(1, 3), ylim=(-17, -13), xticks=[1, 1.5, 2, 2.5],
           title=f'step {step}: count $\\pm$ {EPS_N[t]}, distance $\\pm$ {EPS_D[t]:.0%}\n{smc_nsims[t]:,} simulations so far')
    if show:
        plt.tight_layout(); plt.show()

def smc_controls():
    controls_right(plot_smc, step=IntSlider(value=1, min=1, max=len(EPS_N), step=1, description='step'))


In [ ]:
# More slide functions (a "skip" cell): the hover graph, the prior, checking the fit, checking the intervals
import io, base64, html
from IPython.display import HTML
from ipywidgets import ToggleButtons

# ---- the kilonova graph next to the simulator's code. Hover a circle (or a line) to see that circle's values in
# our simulated year and light up its line. Pure CSS (:has + :hover), no JavaScript - the same trick as last Thursday.
def _png(draw):
    fig, ax = plt.subplots(figsize=(4.4, 2.2)); draw(ax); ax.set_yticks([]); plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=120, facecolor='white'); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()
def _kn_minis():
    m_all = M_all + 5 * np.log10(r_all * 1e6) - 5
    V = F_OBS * 4 / 3 * np.pi * 1.5**3                                  # Gpc^3: half the sky out to 1.5 Gpc
    def R_ax(ax):
        ax.hist(10**log_R, bins=np.logspace(1, 3, 40), color='0.6', label='prior: flat in log R')
        ax.axvline(R_TRUE, color='C3', lw=2, label=f'true R = {R_TRUE:.0f}'); ax.set_xscale('log')
        ax.set_xlabel('R (Gpc$^{-3}$ yr$^{-1}$)'); ax.legend(fontsize=9, loc='upper right', frameon=False)
    def mu_ax(ax):
        ax.hist(mu_draw, bins=40, color='0.6', density=True, label='prior: flat'); ax.axvline(MU_TRUE, color='C3', lw=2, label=f'true $\\mu$ = {MU_TRUE:.1f}')
        ax.set_xlabel('$\\mu$ (mag)'); ax.invert_xaxis(); ax.legend(fontsize=9, loc='lower left', frameon=False)
    def sm_ax(ax):
        x = np.linspace(MU_TRUE - 2, MU_TRUE + 2, 200); ax.plot(x, np.exp(-0.5 * ((x - MU_TRUE) / SIG_M)**2), 'k', lw=2)
        ax.set_xlabel('M (mag): the spread 0.5 is fixed, not fit'); ax.invert_xaxis()
    def N_ax(ax):
        k = np.arange(250, 560); lam = R_TRUE * V
        ax.plot(k, stats.poisson(lam).pmf(k), 'k', lw=2, label=f'Poisson(R x volume), mean {lam:.0f}')
        ax.axvline(len(r_all), color='C0', lw=2, label=f'this year: {len(r_all)} mergers'); ax.set_xlabel('N, mergers out to 1.5 Gpc')
        ax.legend(fontsize=8, loc='center right', frameon=False)
    def M_ax(ax):
        b = np.arange(-17, -12.5, 0.2)
        ax.hist(M_all, b, color='0.75', label='all'); ax.hist(M_all[found], b, color='C0', label='found')
        ax.set_xlabel('peak absolute mag $M_j$'); ax.invert_xaxis(); ax.legend(fontsize=9, loc='upper left', frameon=False)
    def d_ax(ax):
        b = np.linspace(0, 1500, 31)
        ax.hist(r_all, b, color='0.75', label='all'); ax.hist(r_all[found], b, color='C0', label='found')
        ax.set_xlabel('distance $d_j$ (Mpc)'); ax.legend(fontsize=9, loc='upper left', frameon=False)
    def m_ax(ax):
        b = np.arange(18, 30, 0.25)
        ax.hist(m_all, b, color='0.75', label='all'); ax.hist(m_all[found], b, color='C0', label='found')
        ax.axvline(M_LIM, color='k', ls='--', lw=1.5, label='cut, 24.0'); ax.set_xlabel('apparent peak mag $m_j$')
        ax.legend(fontsize=9, loc='upper left', frameon=False)
    def I_ax(ax):
        ax.bar([0, 1], [(~found).sum(), found.sum()], color=['0.75', 'C0'], width=0.6)
        ax.set_xticks([0, 1], [f'$I_j$ = 0: missed ({(~found).sum()})', f'$I_j$ = 1: found ({found.sum()})']); ax.set_xlim(-0.6, 1.6)
    def Nd_ax(ax):
        k = np.arange(5, 60); lam = expected_count(R_TRUE, MU_TRUE)
        ax.plot(k, stats.poisson(lam).pmf(k), 'k', lw=2, label=f'expected {lam:.0f}'); ax.axvline(N_obs, color='C0', lw=2, label=f'we found {N_obs}')
        ax.set_xlabel('$N_{\\rm det}$'); ax.legend(fontsize=9, loc='upper right', frameon=False)
    return {k: _png(f) for k, f in [('R', R_ax), ('mu', mu_ax), ('sm', sm_ax), ('Nt', N_ax), ('Mj', M_ax), ('d', d_ax),
                                    ('mj', m_ax), ('I', I_ax), ('thr', m_ax), ('Nd', Nd_ax)]}
_KN_CODE = [('Nt',   "n = rng.poisson(R * volume)              # N: how many mergers"),
            ('d',   "d = r_out * rng.uniform(size=n)**(1/3)   # d_j: even in volume"),
            ('Mj',  "M = rng.normal(mu, 0.5, n)               # M_j: peak abs. mag"),
            ('mj',  "m = M + 5 * np.log10(d * 1e6) - 5        # m_j: apparent mag"),
            ('I',   "found = m <= 24.0                        # I_j: the cut"),
            ('Nd',  "N_det = found.sum()                      # N_det")]
_LINE_OF = {'R': 'Nt', 'mu': 'Mj', 'sm': 'Mj', 'thr': 'I'}   # keys differ by more than case            # circles with no line of their own light up their child's line
_KN_NODES = {'mu': (55, 45, 'μ', 'blue'), 'sm': (160, 45, '0.5', 'dot'), 'R': (320, 45, 'R', 'blue'),
             'Mj': (95, 145, 'M<tspan font-size="13" dy="5">j</tspan>', 'white'), 'd': (205, 145, 'd<tspan font-size="13" dy="5">j</tspan>', 'grey'),
             'Nt': (320, 145, 'N', 'white'), 'mj': (95, 245, 'm<tspan font-size="13" dy="5">j</tspan>', 'grey'),
             'I': (205, 245, 'I<tspan font-size="13" dy="5">j</tspan>', 'grey'), 'Nd': (320, 245, 'N<tspan font-size="13" dy="5">det</tspan>', 'grey'),
             'thr': (255, 322, 'm ≤ 24.0', 'dot')}
def kn_pgm_hover():
    minis = _kn_minis(); k = list(_KN_NODES)
    css = '.knw .pop{display:none} .knw .nd{cursor:pointer} .knw pre{font-size:14px;line-height:1.7;margin:0} .knw .ln{display:block;padding:0 4px}'
    for n in k:
        on = f'.knw:has(.n-{n}:hover, .ln-{n}:hover)'
        css += f' {on} .pop-{n}{{display:block}} {on} .ln-{_LINE_OF.get(n, n)}{{background:#fde68a}} {on} .n-{n} circle{{stroke:#f59e0b;stroke-width:5}}'
    css += ' .knw:has(.nd:hover, .ln:hover) .hint{display:none}'
    svg = ['<svg width="370" height="360" viewBox="0 0 370 360" font-family="sans-serif">',
           '<defs><marker id="ah20" markerWidth="10" markerHeight="8" refX="9" refY="4" orient="auto"><path d="M0,0 L10,4 L0,8 z"/></marker></defs>',
           '<rect x="45" y="98" width="215" height="202" fill="none" stroke="black"/><text x="52" y="294" font-size="15">j = 1…N</text>']
    for a, b in [('mu', 'Mj'), ('sm', 'Mj'), ('Mj', 'mj'), ('d', 'mj'), ('mj', 'I'), ('thr', 'I'), ('R', 'Nt'), ('Nt', 'Nd'), ('I', 'Nd')]:
        (x1, y1, *_), (x2, y2, *_) = _KN_NODES[a], _KN_NODES[b]; L = np.hypot(x2 - x1, y2 - y1)
        r1 = 6 if _KN_NODES[a][3] == 'dot' else 30
        svg.append(f'<line x1="{x1 + (x2 - x1) * r1 / L:.0f}" y1="{y1 + (y2 - y1) * r1 / L:.0f}" x2="{x2 - (x2 - x1) * 32 / L:.0f}" y2="{y2 - (y2 - y1) * 32 / L:.0f}" stroke="black" stroke-width="1.5" marker-end="url(#ah20)"/>')
    for n, (x, y, lab, kind) in _KN_NODES.items():
        if kind == 'dot':
            ty = y - 14 if n == 'sm' else y + 26
            svg.append(f'<g class="nd n-{n}"><circle cx="{x}" cy="{y}" r="16" fill="white" stroke="white"/><circle cx="{x}" cy="{y}" r="6" fill="black"/><text x="{x}" y="{ty}" text-anchor="middle" font-size="18">{lab}</text></g>')
        else:
            fill = {'blue': 'white', 'white': 'white', 'grey': '#b3b3b3'}[kind]; ec = '#1f77b4' if kind == 'blue' else 'black'
            svg.append(f'<g class="nd n-{n}"><circle cx="{x}" cy="{y}" r="30" fill="{fill}" stroke="{ec}" stroke-width="{3 if kind == "blue" else 1.5}"/><text x="{x}" y="{y + 7}" text-anchor="middle" font-size="22" font-style="italic">{lab}</text></g>')
    svg.append('</svg>')
    code_html = ''.join(f'<span class="ln ln-{n}">{html.escape(t)}</span>' for n, t in _KN_CODE)
    pops = ''.join(f'<img class="pop pop-{n}" src="data:image/png;base64,{minis[n]}" style="width:400px">' for n in k)
    display(HTML(f'<style>{css}</style><div class="knw" style="display:flex;gap:24px;align-items:flex-start">'
                 f'<div>{"".join(svg)}</div><div><pre>{code_html}</pre><div style="height:210px;margin-top:10px">'
                 f'<div class="hint" style="color:#666;padding-top:60px">hover a circle or a line to see its values in our simulated year</div>{pops}</div></div></div>'))

# ---- does the prior matter? Count-only ABC (mu held at the truth) after the first few months of the survey
t_found = np.sort(np.random.default_rng(3).uniform(0, 12, N_obs))     # month each of our 29 was found (made up, even in time)
def plot_prior(prior='flat in log R', months=12):
    n = int((t_found < months).sum()); f = months / 12
    rng2 = np.random.default_rng(8)
    R2 = 10**rng2.uniform(1, 3, 200_000) if prior == 'flat in log R' else rng2.uniform(10, 1000, 200_000)
    keep2 = rng2.poisson(expected_count(R2, MU_TRUE, years=f)) == n
    lg = np.linspace(1, 3, 400); like = stats.poisson(expected_count(10**lg, MU_TRUE, years=f)).pmf(n)
    post = like * (1 if prior == 'flat in log R' else 10**lg); post /= np.trapezoid(post, lg)
    other = like * (10**lg if prior == 'flat in log R' else 1); other /= np.trapezoid(other, lg)
    pr = np.ones_like(lg) if prior == 'flat in log R' else 10**lg; pr /= np.trapezoid(pr, lg)
    fig, ax = plt.subplots(figsize=(7.4, 3.5))
    ax.hist(np.log10(R2[keep2]), bins=np.linspace(1, 3, 51), density=True, color='C0', alpha=0.6, label=f'ABC: kept {keep2.sum():,}')
    ax.plot(lg, post, 'k', lw=2, label='exact posterior')
    ax.plot(lg, other, color='0.5', ls='--', lw=1.5, label='exact, with the other prior')
    ax.plot(lg, pr, color='C2', ls='-.', lw=1.5, label='this prior')
    ax.axvline(np.log10(R_TRUE), color='C3', ls=':', label='true R')
    ax.set(xlabel='log$_{10}$ R', ylabel='density', xlim=(1, 3), title=f'first {months} month(s): {n} found, prior {prior}')
    ax.legend(fontsize=9, loc='upper right'); plt.tight_layout(); plt.show()
def prior_controls():
    controls_right(plot_prior, prior=ToggleButtons(options=['flat in log R', 'flat in R'], value='flat in R'),
                   months=IntSlider(value=1, min=1, max=12, step=1, description='months'))

# ---- posterior predictive check: simulate a new year from each kept (R, mu), and compare with ours
def plot_ppc():
    rng3 = np.random.default_rng(11); k = np.where(keep_Nd)[0]
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.5))
    for i in k:
        d, m = simulate(10**log_R[i], mu_draw[i], rng3)
        ax[0].step(np.sort(d), np.arange(1, len(d) + 1) / len(d), color='C0', alpha=0.15, lw=1)
        ax[1].step(np.sort(m), np.arange(1, len(m) + 1) / len(m), color='C0', alpha=0.15, lw=1)
    ax[0].step(np.sort(d_obs), np.arange(1, N_obs + 1) / N_obs, 'k', lw=2.5)
    ax[1].step(np.sort(m_obs), np.arange(1, N_obs + 1) / N_obs, 'k', lw=2.5)
    ax[0].set(xlabel='distance (Mpc)', ylabel='fraction found closer', title='used by ABC (its median)')
    ax[1].set(xlabel='apparent peak mag $m$', ylabel='fraction found brighter', title='not used by ABC', xlim=(20, 24.1))
    ax[0].legend([Line2D([], [], color='C0'), Line2D([], [], color='k', lw=2.5)], [f'{len(k)} new years, one per kept draw', 'our data'], fontsize=9, loc='lower right')
    plt.tight_layout(); plt.show()

# ---- coverage: pretend each of 400 prior simulations is "the data", run ABC on it with the other simulations,
# and ask how often the true values land inside the 50%, 68%, 90% and 95% intervals
def run_coverage(n_test=400, seed=4):
    rng4 = np.random.default_rng(seed)
    pool = np.where((S_sim[:, 0] >= 10) & (S_sim[:, 0] <= 60))[0]
    u = []
    for t in rng4.choice(pool, n_test, replace=False):
        k = (S_sim[:, 0] == S_sim[t, 0]) & (np.abs(np.log(S_sim[:, 2] / S_sim[t, 2])) <= TOL_DIST); k[t] = False
        if k.sum() >= 20:
            u.append([(log_R[k] < log_R[t]).mean(), (mu_draw[k] < mu_draw[t]).mean()])
    return np.array(u)
cov_u = run_coverage()                                                  # about 2 s
LEVELS = np.array([0.5, 0.68, 0.9, 0.95])
def plot_coverage():
    lv = np.linspace(0.01, 0.99, 99)
    fig, ax = plt.subplots(figsize=(5.6, 3.9))
    for j, (name, c) in enumerate([('R', 'C0'), ('$\\mu$', 'C1')]):
        ax.plot(lv, [(np.abs(cov_u[:, j] - 0.5) <= L / 2).mean() for L in lv], c, lw=2, label=name)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect')
    ax.set(xlabel='interval we quote (e.g. 0.95 = 95%)', ylabel='how often the truth is inside', xlim=(0, 1), ylim=(0, 1),
           title=f'ABC on {len(cov_u)} simulated surveys with known truth')
    ax.legend(fontsize=10, loc='upper left'); plt.tight_layout(); plt.show()


## What did the detector do to last Thursday's sample?

<img src="images/day19_static_malmquist_widget.png" alt="Last Thursday's Malmquist widget: simulated sources spread evenly through a volume, log luminosity against distance; the flux limit is a dashed curve; detected sources in blue above it, missed ones in grey below; the mean of the detected ones climbs above the true mean far away" style="display:block;margin:0 auto;max-height:280px;max-width:100%">

* $N$ objects exist, and we find only $N_{\rm det}$ of them - each one gets a flag, found (1) or missed (0)
* far away, only the bright ones get a 1 (i.e. the found sample is brighter than the population)
* for the black holes we wrote $p_{\rm det}$ (the chance of a 1) as a formula, and put it and its average $\alpha$ in the likelihood
* for real LIGO data, $p_{\rm det}$ isn't a formula - they inject fake signals into real data

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If you can write down $p_{\rm det}$, the selection goes into the likelihood.**

</div>

## Today - what if you can't write the likelihood at all?

* kilonovae, and how many Rubin should find
* last Thursday's graph, run forward, as a simulator
* rejection ABC (approximate Bayesian computation), checked against an exact answer
* the tolerance, the prior and the summary, and how to check the answer
* wasting fewer simulations (ABC-SMC), and two uses in cosmology
* next: simulation-based inference with neural networks

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Why not just simulate a whole bunch of events, and check if they look like your data?**

</div>

## What is a kilonova?

<img src="images/day20_eso1733_kilonova_pair.jpg" alt="Left: an artist's impression of two neutron stars merging, a bright flash with blue ejecta. Right: brightness against wavelength from 0.5 to 2 microns for the kilonova in NGC 4993, one line per night from 0.5 to 11.5 days; the blue end fades fast while the near-infrared fades slowly" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Left: ESO/L. Calçada/M. Kornmesser (artist's impression). Right: ESO/Tanvir et al., the kilonova AT2017gfo over 12 days. Both CC BY 4.0.</div>

* when two **neutron stars** merge (LIGO hears these too), the debris builds heavy elements like gold and platinum
* their radioactive decay makes it glow for about a week - we call this a **kilonova**
* see how the blue light fades within days, while the infrared lasts longer (right, one line per night)
* we've seen exactly one with gravitational waves - AT2017gfo, 40 Mpc away (GW170817, Aug 2017)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Apart from AT2017gfo, every kilonova in today's class is simulated.**

</div>

## Rubin has about 4 days to catch one

<img src="images/day20_perkins2026_fig2_g.png" alt="Perkins et al. 2026 Fig. 2, g band only: simulated apparent g magnitude at 160 Mpc against days since the merger; the orange line is the median kilonova with a 16-84 percent band, purple adds a nearly on-axis afterglow; the grey line at 25.0 is Rubin's 5-sigma depth; the kilonova peaks near 21.5 within a day and falls below 25.0 after about 4 days" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Perkins et al. 2026, arXiv:2602.13471 (CC BY 4.0), Fig. 2 (g band; simulated).</div>

* the orange line is a simulated kilonova 160 Mpc away in Rubin's $g$ band (purple adds a jet pointed at us - we leave it out today)
* it's above the grey line (25.0, the faintest one visit can see) for about 4 days
* Rubin comes back every few days, and a kilonova can fade between two visits

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We can't write $P(D \mid \theta)$ for their simulation, but we can draw fake $D$ from it.**

</div>

## Haille Perkins and Brian Fields

<img src="images/day20_perkins_fields.jpg" alt="Two photos: left, Haille Perkins standing by a tree on campus; right, Brian Fields in a suit beside the brass telescope in the Astronomy building" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Left: Haille Perkins. Right: Brian Fields. Illinois Astronomy directory photos.</div>

* Haille is a graduate student here in Astronomy, and Brian is a professor of astronomy and physics (Ved Shah, Genevieve Schroeder and I are the other authors)
* they asked how many kilonovae Rubin will find, by simulating each one's light against the typical gaps between Rubin's visits
* their simple formula gives about 29 a year, and ours copies it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Today we build a much simpler version of their forecast, then run it backwards to learn the rate.**

</div>

## The count is a rate times a volume

* mergers happen at a rate $R$ per Gpc$^3$ per year (Perkins et al. adopt 56)
* each kilonova peaks at absolute magnitude (its brightness seen from 10 pc) $M \sim \mathcal{N}(\mu, 0.5)$
* we set $\mu = -14.8$ to match their 29 a year
* we find a kilonova if its peak apparent magnitude is $m \leq 24.0$ (EXTRA: why not Rubin's 25.0)
* brighter kilonovae can be found farther away, i.e. in a bigger volume (last Thursday's VT)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**I picked a version with a likelihood on purpose - we can check ABC against the exact answer.**

</div>

## In one simulated year, Rubin finds 29 and misses 375

In [ ]:
# every merger out to 1.5 Gpc in one simulated year (grey), the ones Rubin finds (blue), and their magnitudes
plot_one_year()

* far away, only the bright ones get above the dashed cut (left) - that's the Malmquist bias from last Thursday
* the 29 found (right) are our data for today, and most are just above the cut

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The data are a count, apparent magnitudes and (if we take a spectrum of each host galaxy) distances.**

</div>

## Last Thursday's graph for kilonovae

<img src="images/day20_pgm_gw_vs_kn.png" alt="Two graphs side by side. Left, last Thursday's: mu and sigma_pop point to theta_j, which with d_j points to the data D_j, which with the SNR 8 threshold sets the flag I_j; N and the flags set N_det. Right, today's: mu and the fixed 0.5 point to the peak magnitude M_j, which with d_j points to the apparent magnitude m_j, which with the cut m &lt;= 24.0 sets I_j; R points to N, and N and the flags set N_det" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Left: last Thursday's black holes. Right: today's kilonovae. Blue: what we infer. White: never seen. Grey: measured (for the ones we find). Dot: fixed.</div>

* each kilonova's peak magnitude $M_j$ comes from the population $\mathcal{N}(\mu, 0.5)$, as $\theta_j$ did
* we measure its apparent magnitude $m_j = M_j + 5 \log_{10}(d_j / 10\,{\rm pc})$ and, from its host galaxy, its distance $d_j$
* $I_j = 1$ if $m_j \leq 24.0$, and $N_{\rm det}$ counts the 1s

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**It's last Thursday's graph, with $N$ drawn from a rate $R$ and the distances measured.**

</div>

## Running the graph forward


In [ ]:
# hover a circle (or a line of code) to see its values in our simulated year
kn_pgm_hover()

<div style="display:flex; align-items:center; justify-content:center; gap:1.5em;">
<img src="images/day20_pgm_kilonova.png" alt="Today's kilonova graph: mu and the fixed 0.5 point to M_j; M_j and d_j point to m_j; m_j and the cut m &lt;= 24.0 point to I_j; R points to N; N and I_j point to N_det" style="max-height:260px;">
<pre style="font-size:0.85em;"># volume: half the sky, out to 1.5 Gpc
N = poisson(R * volume)                  # R -&gt; N
for j in 1..N:
    d[j] = random distance in the volume  # d_j
    M[j] = normal(mu, 0.5)                # mu -&gt; M_j
    m[j] = M[j] + 5 log10(d[j] / 10 pc)   # M_j, d_j -&gt; m_j
    I[j] = (m[j] &lt;= 24.0)                 # m_j -&gt; I_j
N_det = sum(I)                            # I_j -&gt; N_det</pre>
</div>

(static copy of the hover graph, for readers on GitHub or without running the notebook)

* last Thursday each arrow needed a probability, and the missed ones needed $\alpha$
* here the simulator makes the missed ones itself (hover each circle)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If you can draw the graph, you can simulate from it, even when you can't write its likelihood.**

</div>

## Can you match our data by hand?

In [ ]:
# pick R and mu, simulate one year, and compare with our 29 (black outline)
by_hand_controls()

<img src="images/day20_static_byhand_widget.png" alt="The by-hand widget at two settings. Left, R = 316 and mu = -14.0: 53 found, far more than our 29, and bunched closer than ours. Right, R = 56 and mu = -14.8: the simulated histograms look like ours" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* fiddle with $R$ and $\mu$ until the orange looks like the black outline (the count and the distances)
* then change the year - the same $R$ and $\mu$ give a different sample every time

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**ABC does exactly this, 200,000 times, and keeps the $(R, \mu)$ that match.**

</div>

## ABC: simulate, compare, keep

<img src="images/day20_abc_flow.png" alt="Rejection ABC as five boxes: draw theta from the prior, run the simulator, get fake data, compute its summary, keep theta if the fake summary is within epsilon of the same summary of our data; repeat many times and the kept theta are draws from the posterior given the summary" style="display:block;margin:0 auto;max-height:220px;max-width:100%">

* we reduce the data to a few numbers (the count, a median, like week 1's robust summaries) - we call this a **summary statistic** $S(D)$
* how close counts as a match is called the **tolerance** $\epsilon$ (not last Thursday's leapfrog step)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Draw $\theta$, simulate, and keep the $\theta$ where the simulation matches your real data.**

</div>

## Rejection ABC

```
while we want more samples:
    draw theta from the prior
    simulate fake data from theta, and compute its summary S_sim
    if S_sim is within eps of S_obs: keep theta
    otherwise: reject theta
```

* let's start simple and hold $\mu$ at its true value (i.e. $\theta$ is just $R$)
* the summary is the count $N_{\rm det}$, and the prior is flat in $\log R$ from 10 to 1000
* here we can also compute the exact posterior, and check ABC against it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**ABC swaps the likelihood for a simulator, a summary and a tolerance (you still need a prior!)**

</div>

## With $\epsilon = 0$, ABC gives the exact posterior

In [ ]:
# 200,000 values of R from the prior; keep the ones whose simulated count is within epsilon of ours
eps_controls()

<img src="images/day20_static_eps_widget.png" alt="The epsilon widget at two settings. Left, epsilon = 0: the histogram of kept R values in blue lies on the black exact posterior, which peaks near 50; the true R = 56 is a red dotted line. Right, epsilon = 20: the histogram is much wider than the exact posterior" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* an exact count match loses nothing, but keeps only 0.7% of the draws
* (the likelihood only uses the count, i.e. the count is a *sufficient* summary - more on that below)
* slide $\epsilon$ up to 20, and 38% are kept, with the histogram broader, back toward the prior

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The smaller $\epsilon$ is, the closer you get to the exact posterior, and the more simulations you reject.**

</div>

## You still need a prior

In [ ]:
# count-only ABC after the first few months of the survey, with two different priors
prior_controls()

<img src="images/day20_static_prior_widget.png" alt="The prior widget, prior flat in R, at two settings. Left, after 1 month (2 found): the ABC histogram and the exact posterior sit well to the right of the dashed curve for the prior flat in log R. Right, after 12 months (29 found): both posteriors are narrow and almost on top of each other" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* pretend we only have the first few months of the survey (our 29, spread evenly through the year)
* after 1 month (2 found), flat in $R$ and flat in $\log R$ give answers 1.5x apart (dashed: the other prior)
* after 12 months the two agree, because the data now outweigh the prior

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**With few events the prior matters, because ABC draws every $\theta$ from it.**

</div>

## Can the count alone tell us both $R$ and $\mu$?

* now let's infer both, $\theta = (R, \mu)$, with a flat prior on $\mu$ from $-17$ to $-13$
* we run 200,000 simulations and keep the ones with exactly 29 found

In [ ]:
# kept draws (blue) out of the prior draws (grey), with the truth as a red star
plot_ridge()

* the kept draws lie along a ridge, i.e. twice the rate with kilonovae 0.5 mag fainter gives the same count

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Notice that the count alone can't separate $R$ from $\mu$.**

</div>

## Of the two medians, only the distance changes with $\mu$

* each dot below is one simulation from the prior, and the dashed line is our data

In [ ]:
# median apparent magnitude (left) and median distance (right) of the kilonovae found, against mu
plot_summaries()

* make every kilonova 1 mag brighter, and Rubin sees 1.6x farther
* but the ones found are still just above the cut, and their median magnitude barely changes (left)
* their median distance grows (right)
* changing $R$ only changes how many we find

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Pick summaries that respond differently to each parameter (here, the distances)!**

</div>

## You choose the summaries

In [ ]:
# tick a summary to add it; the distance tolerance is a slider
choose_controls()

<img src="images/day20_static_summary_widget.png" alt="The summary widget at three settings: with the count only, the kept draws lie along a diagonal ridge; adding the median magnitude keeps the same ridge; adding the median distance leaves a small blob around the true value" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* a match is the same count, plus whichever you tick (median magnitude within 0.15 mag, median distance within 5%)
* tick the magnitude and you still get the ridge (662 of the 1193 count-only matches)
* tick the distance and you get $R$ = 57 (28-98 at 95%) and $\mu$ = -14.83, cf. truth 56 and -14.8
* widen the distance tolerance, and the blob grows back toward the ridge

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Choose your summary statistics carefully - they're the only part of the data ABC ever sees!**

</div>

## ABC gives the posterior given the summary

<img src="images/day15_datasaurus_dozen.gif" alt="The Datasaurus Dozen from Day 15: an animation of thirteen scatter plots, a dinosaur, a star, a circle, lines and crosses, all with the same means, standard deviations and correlation" style="display:block;margin:0 auto;max-height:220px;max-width:100%">

* ABC returns $p(\theta \mid S(D))$, which equals $p(\theta \mid D)$ only if the summary keeps everything the data say about $\theta$
* a summary like that is called **sufficient** (the count is, for $R$ alone)
* the Datasaurus from Day 15 (above) has 13 datasets with the same means, SDs and correlation, and ABC on those numbers would keep all 13
* $\epsilon > 0$ is a second approximation (the posterior comes out broader than it should)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**ABC approximates the posterior in the summary and in the tolerance.**

</div>

## Do the kept $(R, \mu)$ make data like ours?

In [ ]:
# a new simulated year from each kept draw (blue), with our data (black)
plot_ppc()

* for each of the 83 kept draws (count + median distance), simulate a new year
* our data (black) are inside the blue for the distances, and also for the magnitudes, which ABC never used
* this is the posterior predictive check from Days 12 and 15, and it works with or without a likelihood

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Check the fit on the parts of the data your summaries left out.**

</div>

## ABC's intervals pass a coverage test

In [ ]:
# run the same ABC on fake surveys whose true (R, mu) we know
plot_coverage()

* pretend each of 397 simulations is the data, and run the same ABC on it with the rest
* we know the truth for each, and count how often it's inside the interval ABC quotes
* 95% intervals hold the truth 95% ($R$) and 93% ($\mu$) of the time - both lines follow the dashed one
* count-only ABC also passes, with much wider $\mu$ intervals - coverage can't tell you a summary is weak

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A 95% interval should hold the truth 95% of the time - you can test that with a simulator.**

</div>

## Rejection ABC runs into the curse of dimensionality

<img src="images/day20_abc_cost.png" alt="Fraction of the 200,000 simulations kept against the distance tolerance, log scale: the count alone keeps about 0.6 percent; adding the median distance keeps far fewer at a 5 percent tolerance, and adding the magnitude as well keeps fewer still" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

* with $R$ and $\mu$ both free, the count alone kept 0.60% of the 200,000 simulations
* adding the median distance kept 0.04%
* each extra summary, or a smaller $\epsilon$, keeps fewer (the curse of dimensionality from Days 12-13, now in the summaries)
* each of our runs takes a few microseconds, but a real survey simulator doesn't

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Rejection ABC wastes most of its simulations.**

</div>

## ABC-SMC (sequential Monte Carlo) shrinks $\epsilon$ in steps

In [ ]:
# each step keeps 1000 draws; move the slider to tighten the tolerance
smc_controls()

<img src="images/day20_static_smc_widget.png" alt="The ABC-SMC widget at steps 1, 3 and 6: the kept draws start as a wide band along the ridge, shrink, and end as a small blob around the true value" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* step 1 is rejection ABC with a loose $\epsilon$
* each later step perturbs the last step's kept draws, and uses a smaller $\epsilon$
* each new draw gets a weight that puts the prior back, because it no longer comes from the prior
* the last step uses our rule from before (same count, distance within 5%)
* it agrees with rejection ABC, $R$ = 55 (30-98) against 57 (28-98), $\mu$ = -14.81 against -14.83
* rejection ABC needed about 2,400 simulations per kept draw, and ABC-SMC about 280

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**ABC-SMC focuses its simulations where the posterior is non-tiny.**

</div>

## ABC on supernova cosmology

<img src="images/day20_weyant2013_fig11.png" alt="Weyant et al. 2013 Fig. 11: 95 percent contours of w against Omega_M. Black dotted: chi-squared fit to type Ia supernovae only. Red dashed: chi-squared fit after adding non-Ia supernovae; it is shifted. Blue solid: ABC on the contaminated sample; it is wider and encloses the black contour" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Weyant, Schafer &amp; Wood-Vasey 2013, ApJ 764, 116 (arXiv:1206.2563), Fig. 11.</div>

* they fit $\Omega_M$ (how much matter) and $w$ (how dark energy behaves) to SDSS type Ia supernovae
* they mix in simulated non-Ia supernovae (25%), the impostors
* a $\chi^2$ fit has no term for the impostors, and its contour (red) is shifted from the clean answer (black)
* their simulator makes impostors too, and ABC accounts for them

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note how the ABC contour (blue) is wider, but still encloses the clean answer!**

</div>

## ABC-SMC on galaxy clusters

<img src="images/day20_ishida2015_fig4.png" alt="Ishida et al. 2015 Fig. 4, one plane from four frames of the animation: heat maps of ABC kept draws in Omega_c (x, 0 to 0.6) against sigma_8 (y, 0.5 to 1.0); the flat prior box shrinks over steps t = 0, 3 and 8 into a thin diagonal band" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Adapted from Ishida et al. 2015, Astronomy &amp; Computing 13, 1 (arXiv:1504.06129), Fig. 4 (one panel from four frames of the animation; bright = many kept draws).</div>

* they infer three cosmological parameters from a simulated catalogue of galaxy clusters, with their public code `cosmoabc`
* the plane shown is $\Omega_c$ (how much dark matter) against $\sigma_8$ (how clumpy matter is)
* after the prior (left), each panel is one step of ABC-SMC (t = 0, 3, 8, counting from 0), and the box shrinks into a thin banana
* the banana is a degeneracy, like our $R$-$\mu$ ridge

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**This is the same ABC-SMC as our slider, with three parameters and a cosmology simulator.**

</div>

## Next, we'll learn from every simulation

<img src="images/day20_lueckmann2021_algorithms.png" alt="Four flowcharts from Lueckmann et al. 2021: Monte Carlo ABC (prior, proposal, simulator, compare with data, posterior) next to three methods that train an estimator on the simulations - likelihood estimation, posterior estimation and ratio estimation - each with a dotted sequential loop" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Lueckmann, Boelts, Greenberg, Gonçalves &amp; Macke 2021, "Benchmarking Simulation-Based Inference", AISTATS (arXiv:2101.04653), Fig. 1; via A. Malz, LSSTC DSFP Session 16.</div>

* today we did the left-most box (its dotted loop is ABC-SMC)
* the other three train a neural network on all the simulations
* it learns the likelihood, the posterior, or the ratio $P(D \mid \theta)/P(D)$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Simulation-based inference keeps the simulator and replaces the rejection step with a network.**

</div>

## What did we learn today?

* even without $P(D \mid \theta)$, you can simulate a whole bunch of events and keep the $\theta$ where they look like your data
* a simulator is a PGM like last Thursday's, run from top to bottom
* rejection ABC draws from the prior, simulates, and keeps $\theta$ when the summaries match within $\epsilon$ (ABC-SMC does it in steps)
* ABC gives $p(\theta \mid S(D))$, which is exact only for a sufficient summary at $\epsilon = 0$
* check it with posterior predictive checks and coverage
* the count of kilonovae can't separate the merger rate from how bright they are, but their distances can

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Check ABC on a problem where you know the answer before you trust it on one where you don't.**

</div>

## Colloquium today, 3:45 pm, 134 Astronomy: Rachel Bezanson (Pitt)

<img src="images/speaker_rachel_bezanson.jpg" alt="Rachel Bezanson, University of Pittsburgh" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

* **"The Beginning of the End - Charting the Emergence and Evolution of Massive Galaxies"**
* she co-leads UNCOVER, a deep JWST survey of the early universe
* refreshments at 3:30 pm in Astronomy 222 (go!)

<div style="font-size:0.8em; text-align:center; color:#666;">Photo: University of Pittsburgh Physics &amp; Astronomy. Listing: astro.illinois.edu/news-events/astrophysics-colloquium; calendars.illinois.edu event 33565766.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A deep survey like UNCOVER is flux-limited - last Thursday's selection problem, at high redshift.**

</div>

## Before you go

* **Quiz 4 is Thu Nov 5** (40 minutes, 20 points, Abha proctors; no lecture that day): GPs, partial pooling, PGMs, selection, and today's ABC
* Lab 08 is due **Wed Nov 4 by Noon** (every date is in `ASSIGNMENTS.md`)
* Lab 09 (ABC and simulation-based inference) is posted Thursday
* EXTRA: where the expected count comes from, and how bright Perkins et al.'s kilonovae are

## EXTRA: where the expected count comes from

* Rubin sees to 25.0 in one visit
* but Perkins et al. assume a kilonova is first caught ~1 mag past its brightest (hence our cut at $m \leq 24.0$ at peak)
* their adopted rate, 56, is uncertain by about a factor of 3 (+99/-40)
* a kilonova of absolute magnitude $M$ is found out to $r_{\rm max} = 10^{(m_{\rm lim} - M)/5 + 1}$ pc, with $m_{\rm lim} = 24.0$
* expected count per year: $\langle N_{\rm det} \rangle = R \, f_{\rm obs} \, \frac{4\pi}{3} \langle r_{\rm max}^3 \rangle$ (Perkins et al. Eq. 6-7), with $f_{\rm obs} = 0.5$ the fraction of sky
* $\langle r_{\rm max}^3 \rangle$ averages over the spread of $M$ and goes as $10^{-3M/5}$ (the bright ones count most)
* check against their Fig. 2: $g \approx 21.5$ at 160 Mpc is $M \approx -14.5$, close to our $\mu = -14.8$

## EXTRA: how bright Perkins et al.'s kilonovae are

<img src="images/day20_perkins2026_fig6_lumfunc.png" alt="Perkins et al. 2026 Fig. 6: probability density of simulated peak absolute magnitude in u and g for kilonovae alone (orange), afterglows alone (teal) and both together (purple, after a colour cut), on a log scale; the kilonova distribution is narrow and peaks near -14 to -15 in g with a long faint tail" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Perkins, Narayan, Fields, Shah &amp; Schroeder 2026, arXiv:2602.13471 (CC BY 4.0), Fig. 6 (simulated).</div>

* their kilonova peak magnitudes are narrow (near $-14$ to $-15$ in $g$) with a long faint tail
* ours is a Gaussian of width 0.5
* the count goes as $\langle 10^{-3M/5} \rangle$ (their Eq. 7)
* most of the kilonovae Rubin finds come from the bright side of this curve